# L8: Advanced Reactor Modeling

This lecture addresses three advanced topics that arise in industrial reactor modeling: (1) **coupled scales** — linking particle-level diffusion-reaction to column-level convection-dispersion; (2) **membrane reactors** — explicit vs implicit coupling strategies; (3) **pressure-velocity coupling** — Darcy flow and the incompressible Navier-Stokes equations. Efficient pymrm implementations exploit the Schur complement and block-diagonal sparsity.

## Governing equations

**Particle-column coupling**

Column: $\varepsilon \frac{\partial c_f}{\partial t} + v\frac{\partial c_f}{\partial z} = D_{ax}\frac{\partial^2 c_f}{\partial z^2} - \frac{3(1-\varepsilon)}{R_p}\left.J\right|_{r=R_p}$

Particle: $\frac{\partial c_p}{\partial t} = \frac{D_p}{r^2}\frac{\partial}{\partial r}\left(r^2\frac{\partial c_p}{\partial r}\right) + R(c_p)$

**Darcy flow** (pressure-velocity coupling):

$$\mathbf{u} = -\frac{K}{\mu}\nabla P, \quad \nabla \cdot \mathbf{u} = 0$$

Combining: $\nabla \cdot \left(\frac{K}{\mu}\nabla P\right) = 0$ — elliptic pressure equation.

## PyMRM building blocks

| Function | Role |
|---|---|
| `construct_interface_matrices` | Maps column ↔ particle fluxes at the interface |
| `clip_approach` | Bound-constrained Newton for concentration limits |
| `stencil_block_diagonals` | Extract block structure for preconditioning |
| `scipy.sparse.bmat` | Assemble block-structured system matrix |
| Schur complement $S = A_{22} - A_{21}A_{11}^{-1}A_{12}$ | Decouple subsystems |

## Example 1 — Particle model coupled to column

A fixed-bed column with particles containing internal diffusion and reaction. Each column cell hosts a 1D spherical particle sub-model.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from pymrm import construct_grad, construct_div

# Column parameters
Lc, Nc = 1.0, 20
dzc = Lc / Nc
zc = (np.arange(Nc) + 0.5) * dzc
v_col, eps = 0.1, 0.4
Dax_col = 1e-3
c_in = 1.0

# Particle parameters
Rp, Np = 1e-3, 10    # radius [m], radial cells
drp = Rp / Np
rp = (np.arange(Np) + 0.5) * drp
Dp_val = 1e-7       # particle diffusivity [m²/s]
kp = 0.1            # particle reaction rate [1/s]

# Particle Thiele modulus and effectiveness factor
phi_p = Rp * np.sqrt(kp / Dp_val)
eta = 3 * (phi_p / np.tanh(phi_p) - 1) / phi_p**2
print(f'Particle: phi = {phi_p:.2f}, eta = {eta:.3f}')

# External mass transfer
kL = 1e-4           # film coefficient [m/s]
ap = 3*(1-eps)/Rp   # specific surface [m²/m³]
kLa = kL * ap

# Pseudo-steady particle model: effective rate constant
# kLa*(cf - cs) = (1-eps)*kp*eta*cs  =>  cs = kLa*cf / (kLa + (1-eps)*kp*eta)
k_eff = kLa * (1-eps)*kp*eta / (kLa + (1-eps)*kp*eta)

# Steady-state column model
off = Dax_col / dzc**2
A_col = sp.diags([-off - v_col/dzc, 2*off + v_col/dzc + k_eff, -off],
                  [-1, 0, 1], shape=(Nc, Nc), format='csr')
rhs_col = np.zeros(Nc)
rhs_col[0] += (off + v_col/dzc) * c_in
c_col = spla.spsolve(A_col, rhs_col)

# Particle profile at column exit (z = Lc)
cs_exit = kLa * c_col[-1] / (kLa + (1-eps)*kp*eta)
phi_rp = phi_p * rp / Rp
c_part = cs_exit * np.sinh(phi_rp) / (rp/Rp * np.sinh(phi_p))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(zc, c_col, 'b-o', ms=4)
ax1.set_xlabel('z (m)'); ax1.set_ylabel('c_fluid'); ax1.set_title('Column profile')
ax2.plot(rp*1e3, c_part, 'r-')
ax2.set_xlabel('r (mm)'); ax2.set_ylabel('c_particle'); ax2.set_title('Particle at column exit')
plt.tight_layout(); plt.show()

Particle: phi = 1.00, eta = 0.939


## Example 2 — Pressure-velocity coupling (Darcy flow)

Solve Darcy's law in 1D: heterogeneous permeability field, compute velocity and pressure drop.

In [2]:
# 1D Darcy: -K/mu * dP/dz = u,  d/dz(K/mu * dP/dz) = 0
# With harmonic-mean permeability at faces

Nd = 100
Ld = 1.0
dzd = Ld / Nd
zd = (np.arange(Nd) + 0.5) * dzd
mu = 1e-3  # [Pa·s]

# Heterogeneous permeability: low-K inclusion in centre
K = np.ones(Nd) * 1e-12  # [m²]
K[30:70] = 1e-14          # low-permeability zone

# Harmonic mean at faces
K_face = 2 * K[:-1] * K[1:] / (K[:-1] + K[1:])
K_face = np.concatenate([[K[0]], K_face, [K[-1]]]) / mu

z_f_d = np.linspace(0, Ld, Nd + 1)

# BCs: P=1 at z=0 (Dirichlet), P=0 at z=L (Dirichlet)
# Use finite-difference tridiagonal with harmonic-mean permeability at faces
K_f = K / mu
off_d = K_f / dzd**2
A_d = sp.diags([-off_d[:-1], 2*off_d, -off_d[1:]],
               [-1, 0, 1], shape=(Nd, Nd), format='csr')
A_d = sp.diags([-K_face[1:-1]/dzd**2,
                (K_face[:-1]+K_face[1:])/dzd**2,
                -K_face[1:-1]/dzd**2],
               [-1, 0, 1], shape=(Nd, Nd), format='csr')
rhs_d = np.zeros(Nd)
P_in, P_out = 1e5, 0.0
rhs_d[0] += K_face[0] / dzd**2 * 2 * P_in
rhs_d[-1] += K_face[-1] / dzd**2 * 2 * P_out

P_sol = spla.spsolve(A_d, rhs_d)
u_face = -K_face * np.diff(np.concatenate([[2*P_in - P_sol[0]], P_sol, [2*P_out - P_sol[-1]]])) / dzd

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
ax1.plot(zd, P_sol/1e5); ax1.set_ylabel('P (bar)'); ax1.set_title('Darcy flow')
ax2.semilogy(zd, K*1e12, 'g-'); ax2.set_ylabel('K (mD)'); ax2.set_xlabel('z (m)')
plt.tight_layout(); plt.show()
print(f'Pressure drop: {P_in:.0f} Pa over L={Ld} m')


Pressure drop: 100000 Pa over L=1.0 m


## Summary

- **Multi-scale coupling**: particle models can be solved per column cell; pass surface concentration as BC, return flux to column
- **Implicit coupling** (solving particle + column simultaneously) is more stable; explicit coupling is simpler but may require sub-stepping
- **Schur complement** $S = A_{22} - A_{21}A_{11}^{-1}A_{12}$ enables efficient block-elimination when one block is cheap to invert
- **Darcy flow**: harmonic-mean permeability at faces ensures mass conservation across discontinuities
- **Pressure-velocity coupling** in the full Navier-Stokes context requires projection methods or SIMPLE